# GPS Track from .raw Files

Extracts lat/lon from NME0 (NMEA) datagrams embedded in Simrad EK80 `.raw` files.

Each NME0 datagram has a 4-byte type tag, 8-byte Windows FILETIME timestamp,
followed by a null-terminated NMEA sentence.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, str(Path('../../').resolve()))

from raw_sensor_integration.raw_parser import read_raw_file

RAW_DIR = Path.home() / 'Downloads' / 'drive-download-20251116T003920Z-1-001'
raw_files = sorted(RAW_DIR.glob('*.raw'))
print(f"Found {len(raw_files)} .raw files:")
for f in raw_files:
    print(f"  {f.name}")


In [ ]:
# Remove the first entry
raw_files = raw_files[1:]

In [ ]:
from raw_sensor_integration.gps_parser import parse_nme0_datagrams

In [ ]:
# Inspect what NMEA sentence types are actually in the files
sample_file = raw_files[0]
print(f"Sampling {sample_file.name}...")
sample_data = read_raw_file(sample_file)
sample_records = parse_nme0_datagrams(sample_data)
print(f"Total NME0 datagrams: {len(sample_records)}")

from collections import Counter
type_counts = Counter(r['sentence'][:6] for r in sample_records)
print("\nNMEA sentence types found:")
for stype, count in type_counts.most_common():
    print(f"  {stype:10s}  {count:6d}x  — {sample_records[next(i for i,r in enumerate(sample_records) if r['sentence'].startswith(stype))]['sentence'][:80]}")

In [ ]:
from raw_sensor_integration.gps_parser import parse_gps_sentences

In [ ]:
# Process all .raw files
all_tracks = []

for raw_file in raw_files:
    print(f"Processing {raw_file.name}...")
    data = read_raw_file(raw_file)
    records = parse_nme0_datagrams(data)
    df = parse_gps_sentences(records)
    if len(df) > 0:
        df['file'] = raw_file.name
        all_tracks.append(df)
        print(f"  → {len(df)} GPS fixes  {df['sentence_type'].value_counts().to_dict()}")
    else:
        print("  → No GPS position sentences found")
        # Show what sentence types were present for diagnosis
        types = Counter(r['sentence'][:6] for r in records)
        print(f"     Sentence types in file: {dict(types.most_common(10))}")

if all_tracks:
    track = pd.concat(all_tracks, ignore_index=True).sort_values('timestamp').reset_index(drop=True)
    print(f"\nTotal GPS fixes across all files: {len(track)}")
    track.head(10)
else:
    print("\nNo GPS fixes found in any file. Check the sentence type inventory above.")
    track = pd.DataFrame(columns=['timestamp', 'lat', 'lon', 'sentence_type', 'file'])

In [ ]:
if len(track) > 0:
    print(track[['lat', 'lon']].describe())
    print(f"\nTime range: {track['timestamp'].min()} → {track['timestamp'].max()}")
    print(f"\nSentence types used:\n{track['sentence_type'].value_counts()}")

In [ ]:
import contextily as cx

if len(track) > 0:
    fig, ax = plt.subplots(figsize=(14, 10), dpi=150)
    colors = plt.cm.tab10.colors
    for i, (fname, group) in enumerate(track.groupby('file')):
        g = group.sort_values('timestamp')
        c = colors[i % len(colors)]
        ax.plot(g['lon'], g['lat'], '-', color=c, linewidth=1.5, alpha=0.85,
                label=fname, zorder=3)
        ax.plot(g['lon'].iloc[0],  g['lat'].iloc[0],  '^', color=c, markersize=8, zorder=5)
        ax.plot(g['lon'].iloc[-1], g['lat'].iloc[-1], 's', color=c, markersize=8, zorder=5)
    cx.add_basemap(ax, crs='EPSG:4326',
                   source=cx.providers.CartoDB.Positron,
                   zoom=13, attribution_size=6)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    ax.set_title('Ship GPS Track  (▲ = start, ■ = end)')
    ax.legend(fontsize=7, loc='upper left', framealpha=0.9, title='File')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
if len(track) > 0:
    out_path = Path('gps_track.csv')
    track.to_csv(out_path, index=False)
    print(f"Saved {len(track)} rows to {out_path.resolve()}")

## GPS Quality Analysis

Derive speed and step distances from consecutive GPS fixes, then visualise coverage, anomalies, and vessel behaviour.

In [ ]:
from raw_sensor_integration.gps_parser import haversine_m

# Load and deduplicate: keep one fix per unique timestamp (prefer GGA)
gps = pd.read_csv('gps_track.csv', parse_dates=['timestamp'])
gps['type_rank'] = gps['sentence_type'].map({'$GPGGA': 0, '$GPRMC': 1, '$GPGLL': 2}).fillna(9)
gps = (gps.sort_values(['timestamp', 'type_rank'])
          .drop_duplicates(subset='timestamp')
          .sort_values('timestamp')
          .reset_index(drop=True))

# Compute step-wise metrics within each file (don't diff across file boundaries)
rows = []
for _, grp in gps.groupby('file', sort=False):
    g = grp.sort_values('timestamp').copy()
    g['dt_s']   = g['timestamp'].diff().dt.total_seconds()
    g['dist_m'] = [float('nan')] + [
        haversine_m(g['lat'].iloc[i-1], g['lon'].iloc[i-1],
                    g['lat'].iloc[i],   g['lon'].iloc[i])
        for i in range(1, len(g))
    ]
    g['speed_kn'] = (g['dist_m'] / g['dt_s']) * (3600 / 1852)  # m/s → knots
    rows.append(g)

gps = pd.concat(rows).reset_index(drop=True)

# Clamp speed outliers (GPS jitter at low dt can spike)
gps.loc[gps['dt_s'] < 0.5, ['dist_m', 'speed_kn']] = float('nan')
gps.loc[gps['speed_kn'] > 20, 'speed_kn'] = float('nan')  # >20 kn = sensor artifact

print(f"Fixes after dedup: {len(gps)}")
print(f"Speed range: {gps['speed_kn'].min():.2f} – {gps['speed_kn'].max():.2f} kn")
print(f"Gaps > 5s within files: {(gps['dt_s'] > 5).sum()}")
gps[['timestamp','file','lat','lon','dt_s','dist_m','speed_kn']].head(5)

In [ ]:
import contextily as cx
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

valid = gps.dropna(subset=['speed_kn'])

fig, ax = plt.subplots(figsize=(14, 10), dpi=150)

norm = Normalize(vmin=0, vmax=valid['speed_kn'].quantile(0.98))
sc = ax.scatter(valid['lon'], valid['lat'],
                c=valid['speed_kn'], cmap='plasma', norm=norm,
                s=4, linewidths=0, zorder=3)

cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron,
               zoom=13, attribution_size=6)

cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='plasma'), ax=ax,
                    fraction=0.025, pad=0.01)
cbar.set_label('Speed (knots)', fontsize=9)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('GPS Track coloured by Speed  —  bright = fast, dark = slow/stopped')
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(12, 3), dpi=130)
palette = plt.cm.tab10.colors

for i, (fname, grp) in enumerate(sorted(gps.groupby('file'))):
    g = grp.sort_values('timestamp')
    t0, t1 = g['timestamp'].iloc[0], g['timestamp'].iloc[-1]
    ax.barh(0, t1 - t0, left=t0, height=0.4,
            color=palette[i % len(palette)], alpha=0.85,
            label=fname.replace('.raw', ''))
    ax.text(t0 + (t1 - t0) / 2, 0,
            fname.replace('D20240131-T', '').replace('.raw', ''),
            ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator())
ax.set_yticks([])
ax.set_xlabel('Time of day (UTC)')
ax.set_title('Recording timeline — 2024-01-31')
ax.grid(axis='x', alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import display

rows = []
for fname, grp in gps.groupby('file'):
    g = grp.sort_values('timestamp')
    dur_s = (g['timestamp'].iloc[-1] - g['timestamp'].iloc[0]).total_seconds()
    rows.append({
        'file': fname.replace('.raw',''),
        'fixes': len(g),
        'duration (min)': round(dur_s / 60, 1),
        'mean speed (kn)': round(g['speed_kn'].mean(), 2),
        'max speed (kn)':  round(g['speed_kn'].max(), 2),
        'total dist (km)': round(g['dist_m'].sum() / 1000, 2),
        'gaps > 5s': int((g['dt_s'] > 5).sum()),
    })

summary = pd.DataFrame(rows).set_index('file')
display(summary.style
    .highlight_max(subset=['mean speed (kn)', 'max speed (kn)', 'total dist (km)'],
                   color='#d4edda')
    .highlight_min(subset=['fixes', 'duration (min)'], color='#fff3cd')
    .format(precision=2))
